In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
# Imports & load data
import pandas as pd
import numpy as np

# point to your filtered file
file_path = '/content/drive/MyDrive/MRP/filtered_news_1_sorted.csv'

# read, parsing the Date column as datetime
df = pd.read_csv(
    file_path,
    parse_dates=['Date'],
    dtype={
        'Article_title': str,
        'Stock_symbol':  str,
        'Textrank_summary': str
    },
    low_memory=False
)

df.head()

,Date,Article_title,Stock_symbol,Textrank_summary
0,2009-04-29 00:00:00+00:00,Going Against the Herd,A,NaN
1,2009-06-01 00:00:00+00:00,super-trades - Two 52 week highs and others ab...,A,NaN
2,2009-07-14 00:00:00+00:00,Skystar Bio-Pharmaceutical Announces Expansion...,A,NaN
3,2009-07-30 00:00:00+00:00,The Gold/Silver Ratio From 1300 to 1900...And Now,A,NaN
4,2009-08-04 00:00:00+00:00,"A Simulation of China's 2,$$$,$$$,$$$,$$$ Dive...",A,NaN


In [ ]:
# Quick info & missing-value rates
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\n% missing per column:")
print((df.isna().mean() * 100).round(2))

Shape: (15549299, 4)

Data types:
 Date                datetime64[ns, UTC]
Article_title                    object
Stock_symbol                     object
Textrank_summary                 object
dtype: object

% missing per column:
Date                 0.00
Article_title        0.00
Stock_symbol        63.06
Textrank_summary    83.97
dtype: float64


In [ ]:
# Text‐length features with NaN handling
for col in ['Article_title', 'Textrank_summary']:
    # turn NaNs into ''
    df[col] = df[col].fillna('')
    # character count
    df[f'{col}_chars'] = df[col].str.len()
    # word count
    df[f'{col}_words'] = df[col].str.split().str.len()

# Show descriptive stats for the new columns
text_stats = (
    df
    .filter(regex='_(chars|words)$')
    .describe()
    .T
)
text_stats

,count,mean,std,min,25%,50%,75%,max
Article_title_chars,15549299.0,64.110428,26.274311,0.0,48.0,59.0,74.0,512.0
Article_title_words,15549299.0,9.834919,3.942125,0.0,7.0,9.0,11.0,77.0
Textrank_summary_chars,15549299.0,99.345004,266.137009,0.0,0.0,0.0,0.0,53908.0
Textrank_summary_words,15549299.0,16.116239,43.175833,0.0,0.0,0.0,0.0,9080.0


In [ ]:
# Categorical summaries
print("\nTop 10 Stock Symbols:\n", df['Stock_symbol'].value_counts().head(10))


Top 10 Stock Symbols:
 Stock_symbol
GILD     12376
NVDA     11862
QQQ      11813
BABA     11625
WFC      11301
INTC     11157
MRK      10774
TSLA     10587
KO       10521
BROGW    10456
Name: count, dtype: int64


In [ ]:
# Date/time breakdowns
print("Date range:", df['Date'].min(), "to", df['Date'].max())

# articles per year
df['Year'] = df['Date'].dt.year
print("\nArticles per year:\n", df['Year'].value_counts().sort_index())

# articles per month (last 12)
df['Month'] = df['Date'].dt.to_period('M')
print("\nArticles last 12 months:\n", df['Month'].value_counts().sort_index().iloc[-12:])

Date range: 1914-09-16 00:00:00+00:00 to 2024-01-09 00:00:00+00:00

Articles per year:
 Year
1914          5
1969          1
1999       3081
2000      16176
2001      21974
2002      22179
2003      21557
2004      24386
2005      30718
2006      36057
2007     464439
2008    1056009
2009     961342
2010    1211112
2011    1586808
2012    1605474
2013    1259792
2014    1256672
2015    1471301
2016     817956
2017     515523
2018     698590
2019     700151
2020     351832
2021     181623
2022     280354
2023     954117
2024         70
Name: count, dtype: int64


/tmp/ipython-input-7-2498192736.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['Month'] = df['Date'].dt.to_period('M')



Articles last 12 months:
 Month
2023-02     27125
2023-03     27595
2023-04     29813
2023-05     34523
2023-06     28415
2023-07     50298
2023-08     85494
2023-09     78438
2023-10     89036
2023-11     93832
2023-12    386072
2024-01        70
Freq: M, Name: count, dtype: int64


In [ ]:
# Aggregate summary table
summary = pd.DataFrame({
    'count':     df.count(),
    'missing%':  (df.isna().mean() * 100).round(2),
    'unique':    df.nunique()
})
summary.loc[['Date','Article_title','Stock_symbol','Textrank_summary']]

,count,missing%,unique
Date,15549299,0.00,2796596
Article_title,15549299,0.00,10452637
Stock_symbol,5744672,63.06,8552
Textrank_summary,15549299,0.00,1700713
